<a href="https://colab.research.google.com/github/dorothea-pudding/114-1_TAICA_homework/blob/main/%E6%9C%9F%E6%9C%AB%E5%B0%88%E9%A1%8C%EF%BC%9A%E9%83%B5%E4%BB%B6%E9%80%B1%E5%A0%B1%E5%B7%A5%E5%85%B7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

建立IMAP、讀入信件內容

一開始我是想用google cloud consle讀mail，但是它要求要登記信用卡，我沒有信用卡，所以改採用IMAP。

In [ ]:
!pip install imapclient
!pip install email_validator

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 182.5/182.5 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 9.0 MB/s eta 0:00:00


In [ ]:
import imaplib
import email
from email.header import decode_header
from google.colab import userdata

# 1. 設定帳號清單
IMAP_SERVER = "imap.gmail.com"

EMAIL_ACCOUNTS = [
    {
        "name": "Personal (個人)", #帳號標籤
        "email": "dorothea.pudding@gmail.com", #填入要讀信產生週報的帳號
        "password": userdata.get('Mail_personal'), #填入Google帳戶的應用程式密碼，由16個字母組成
        "imap_server": "imap.gmail.com" #這個不用動它
    },
    {
        "name": "Personal1 (個人)",
        "email": "pretty.rubychou@gmail.com",
        "password": userdata.get('Mail_persona1'),
        "imap_server": "imap.gmail.com"
    },
    {
        "name": "School (學校)",
        "email": "d1134163001@gm.lhu.edu.tw",
        "password": userdata.get('Mail_school'),
        "imap_server": "imap.gmail.com"
    }

]

# 2. 連線函式
def connect_imap(account):
    try:
        print(f"正在連線到 {account['name']} ({account['email']})...")
        mail = imaplib.IMAP4_SSL(account["imap_server"])
        mail.login(account["email"], account["password"])
        mail.select("inbox")
        return mail
    except Exception as e:
        print(f"連線失敗 [{account['name']}]: {e}")
        return None

# 3. 主執行區塊：迴圈處理每個帳號
for account in EMAIL_ACCOUNTS:
    print("-" * 30)

    # 呼叫函式取得連線物件 (這裡才產生 mail 變數)
    mail = connect_imap(account)

    if mail:
        try:
            # 搜尋未讀郵件 (UNSEEN)
            # 如果要搜尋所有郵件改成 'ALL'，或特定日期 'SINCE "15-Dec-2025"'
            status, messages = mail.search(None, 'UNSEEN')

            if status == 'OK':
                email_ids = messages[0].split()
                print(f"[{account['name']}] 未讀郵件數：{len(email_ids)}")

                # 在這裡可以呼叫你之前寫好的分析函式
                # 例如: if email_ids: process_emails_in_range(mail, email_ids)

            else:
                print(f"[{account['name']}] 搜尋失敗")

        except Exception as e:
            print(f"[{account['name']}] 讀取錯誤: {e}")


------------------------------
正在連線到 Personal (個人) (dorothea.pudding@gmail.com)...
[Personal (個人)] 未讀郵件數：23
------------------------------
正在連線到 Personal1 (個人) (pretty.rubychou@gmail.com)...
[Personal1 (個人)] 未讀郵件數：58
------------------------------
正在連線到 School (學校) (d1134163001@gm.lhu.edu.tw)...
[School (學校)] 未讀郵件數：0


In [ ]:
#設定信箱連線資訊
IMAP_SERVER = "imap.gmail.com"
IMAP_PORT = 993

In [ ]:
#讀取收件匣內的郵件編號
mail.select("inbox")  # 選擇收件匣

# 搜尋所有郵件（ALL）；若想只看未讀可改成 'UNSEEN'
status, data = mail.search(None, "ALL")

email_ids = data[0].split()

print("收件匣郵件總數：", len(email_ids))
print("最近 5 封郵件 ID：", email_ids[-5:])

收件匣郵件總數： 197
最近 5 封郵件 ID： [b'193', b'194', b'195', b'196', b'197']


In [ ]:
#讀取一封郵件內容
import email
from email.header import decode_header

def decode_str(s):
    if s is None:
        return ""
    decoded, charset = decode_header(s)[0]
    if isinstance(decoded, bytes):
        return decoded.decode(charset or "utf-8", errors="ignore")
    return decoded

# 取最新一封
latest_email_id = email_ids[-1]

status, msg_data = mail.fetch(latest_email_id, "(RFC822)")
raw_email = msg_data[0][1]
msg = email.message_from_bytes(raw_email)

# 寄件者
from_ = decode_str(msg["From"])
# 主旨
subject = decode_str(msg["Subject"])

print("寄件者：", from_)
print("主旨：", subject)
print("="*50)

# 讀取內容
body_text = ""

if msg.is_multipart():
    for part in msg.walk():
        if part.get_content_type() == "text/plain":
            charset = part.get_content_charset() or "utf-8"
            body_text += part.get_payload(decode=True).decode(charset, errors="ignore")
else:
    charset = msg.get_content_charset() or "utf-8"
    body_text = msg.get_payload(decode=True).decode(charset, errors="ignore")

print("郵件內容：\n")
print(body_text[:1500])  # 僅印出前 1500 字

寄件者： 龍華科技大學校園訊息發佈
主旨： 【資圖處訊息】圖書館2026「KMOVIE 雲端公播電影網」新片上架 ，放假一起看電影！
郵件內容：

 

2026「KMOVIE 雲端公播電影網」新片上架 ，放假一起看電影！

 

★ 免設定、免安裝、無廣告。

★ 100部Disney、Fox、Sony等國外各大電影片商優質影片，
    例如：

雷霆特攻隊、星際寶貝：史迪奇、
地球特派員、異形：羅穆路斯、
柏靈頓：熊熊去秘魯、猛毒最終章：最後一舞、
獵人克萊文、腦筋急轉彎2、
海洋奇緣2、獅子王：木法沙、
猩球崛起：王國誕生、絕地戰警：生死與共...

★ 在校內直接連線觀看 ；校外透過圖書館"校外連線"，登入後使用。

路徑及網址:

校內： <https://kmovie.twedu.com.tw/> KMOVIE 雲端公播電影網( https://kmovie.twedu.com.tw/)

校外：圖書館首頁/ 館藏資源/ 電子資料庫/中文資料庫/校外連線

 ( <https://il.lhu.edu.tw/p/405-1014-16848,c1397.php?Lang=zh-tw>  https://il.lhu.edu.tw/p/405-1014-16848,c1397.php?Lang=zh-tw )

使用期限：至2027年1月31日

 

＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝

本信件內容由資訊圖書處代為發送，若有任何問題，請直接與承辦單位聯絡，

並請勿直接回覆信件！謝謝您的合作。

請注意，依據行政院及教育部與學術網路管理會等相關規定：

１．請同仁使用學校建置的mail2000電子郵件系統作為公務使用信箱。

２．請勿將公務信件轉置私人信箱，公務信箱也務必不收取私人用途信件。

３．請避免使用學校提供的gmail信箱作為公務使用(宣導期間後，公務信件不再寄發)。

＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝

 

 




In [ ]:
def process_single_account(account, start_date):
    mail = connect_imap(account)
    email_ids = fetch_emails_last_week(mail, start_date)

    account_data = []

    for eid in email_ids:
        result, msg_data = mail.fetch(eid, "(RFC822)")
        raw_email = msg_data[0][1]
        msg = email.message_from_bytes(raw_email)

        subject = decode_str(msg["Subject"])
        text_content = extract_text_from_email(msg)

        importance = analyze_email_importance(subject, text_content)
        summary = summarize_email(subject, text_content)

        account_data.append({
            "account": account["name"],  # ⭐ 重點
            "subject": subject,
            "importance": importance,
            "summary": summary
        })

    mail.logout()
    return account_data


In [ ]:
#讀全部郵件
all_mails = []

for eid in email_ids:
    status, msg_data = mail.fetch(eid, "(RFC822)")
    raw = msg_data[0][1]
    msg = email.message_from_bytes(raw)

    subject = decode_str(msg["Subject"])
    from_ = decode_str(msg["From"])

    all_mails.append({
        "id": eid.decode(),
        "from": from_,
        "subject": subject
    })

print("已讀取全部郵件基本資訊，郵件數：", len(all_mails))

已讀取全部郵件基本資訊，郵件數： 197


將內容轉換成給AI看的格式

In [ ]:
##html 轉文字套件
!pip install beautifulsoup4
!pip install lxml

In [ ]:
#讀取郵件標題
from email.header import decode_header

def decode_email_subject(raw_subject):
    """解碼 Email 標題的輔助函式"""
    if not raw_subject:
        return "(無標題)"
    decoded_list = decode_header(raw_subject)
    subject = ""
    for content, encoding in decoded_list:
        if isinstance(content, bytes):
            subject += content.decode(encoding or "utf-8", errors="ignore")
        else:
            subject += str(content)
    return subject

In [ ]:
#建立郵件解析工具>純文字化
import email
from email.header import decode_header
from bs4 import BeautifulSoup

def decode_str(s):
    if s is None:
        return ""
    decoded, charset = decode_header(s)[0]
    if isinstance(decoded, bytes):
        try:
            return decoded.decode(charset or "utf-8", errors="ignore")
        except:
            return decoded.decode("utf-8", errors="ignore")
    return decoded

def extract_subject_and_text(msg):
    """
    同時提取 Email 的標題 (Subject) 與純文字內文 (Text)
    回傳: (subject, text)
    """
    # 1. 提取並解碼標題
    raw_subject = msg.get("Subject")
    subject = decode_email_subject(raw_subject)

    # 2. 提取內文 (使用原本的邏輯)
    text = ""
    if msg.is_multipart():
        for part in msg.walk():
            content_type = part.get_content_type()
            charset = part.get_content_charset() or "utf-8"

            if part.get_content_disposition() == 'attachment':
                continue

            if content_type == "text/plain":
                try:
                    text += part.get_payload(decode=True).decode(charset, errors="ignore")
                except: pass
            elif content_type == "text/html":
                try:
                    html = part.get_payload(decode=True).decode(charset, errors="ignore")
                    soup = BeautifulSoup(html, "lxml")
                    text += soup.get_text("\n", strip=True)
                except: pass
    else:
        content_type = msg.get_content_type()
        charset = msg.get_content_charset() or "utf-8"
        payload = msg.get_payload(decode=True)
        if payload:
            if content_type == "text/plain":
                text = payload.decode(charset, errors="ignore")
            elif content_type == "text/html":
                html = payload.decode(charset, errors="ignore")
                soup = BeautifulSoup(html, "lxml")
                text = soup.get_text("\n", strip=True)

    # 標準化空白行
    lines = [line.strip() for line in text.split("\n")]
    text = "\n".join([l for l in lines if l])

    # 回傳兩個值：標題, 內文
    return subject, text

In [ ]:
# 取最新郵件
latest_email_id = email_ids[-1]

status, msg_data = mail.fetch(latest_email_id, "(RFC822)")
raw_email = msg_data[0][1]
msg = email.message_from_bytes(raw_email)

subject = decode_str(msg["Subject"])
from_ = decode_str(msg["From"])

# 取得純文字內容
text_content = extract_subject_and_text(msg)

print("寄件者：", from_)
print("主旨：", subject)
print("純文字郵件內容（前 1500 字）：\n")
print(text_content[:1500])


寄件者： 龍華科技大學校園訊息發佈
主旨： 【資圖處訊息】圖書館2026「KMOVIE 雲端公播電影網」新片上架 ，放假一起看電影！
純文字郵件內容（前 1500 字）：

('【資圖處訊息】圖書館2026「KMOVIE 雲端公播電影網」新片上架 ，放假一起看電影！', '2026「KMOVIE 雲端公播電影網」新片上架 ，放假一起看電影！\n★ 免設定、免安裝、無廣告。\n★ 100部Disney、Fox、Sony等國外各大電影片商優質影片，\n例如：\n雷霆特攻隊、星際寶貝：史迪奇、\n地球特派員、異形：羅穆路斯、\n柏靈頓：熊熊去秘魯、猛毒最終章：最後一舞、\n獵人克萊文、腦筋急轉彎2、\n海洋奇緣2、獅子王：木法沙、\n猩球崛起：王國誕生、絕地戰警：生死與共...\n★ 在校內直接連線觀看 ；校外透過圖書館"校外連線"，登入後使用。\n路徑及網址:\n校內： <https://kmovie.twedu.com.tw/> KMOVIE 雲端公播電影網( https://kmovie.twedu.com.tw/)\n校外：圖書館首頁/ 館藏資源/ 電子資料庫/中文資料庫/校外連線\n( <https://il.lhu.edu.tw/p/405-1014-16848,c1397.php?Lang=zh-tw>  https://il.lhu.edu.tw/p/405-1014-16848,c1397.php?Lang=zh-tw )\n使用期限：至2027年1月31日\n＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝\n本信件內容由資訊圖書處代為發送，若有任何問題，請直接與承辦單位聯絡，\n並請勿直接回覆信件！謝謝您的合作。\n請注意，依據行政院及教育部與學術網路管理會等相關規定：\n１．請同仁使用學校建置的mail2000電子郵件系統作為公務使用信箱。\n２．請勿將公務信件轉置私人信箱，公務信箱也務必不收取私人用途信件。\n３．請避免使用學校提供的gmail信箱作為公務使用(宣導期間後，公務信件不再寄發)。\n＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝\n2026\n「\nKMOVIE\n雲端公播電影網」新片上架 ，放假一起看電影！\n★ 免設定、免安裝、無廣告。\n★\n100\n部\

判斷文件重要性

In [ ]:
#載入AI模型
!pip install openai

In [ ]:
#設定API key
from google.colab import userdata
import openai
openai.api_key = userdata.get('Groq')

In [ ]:
#重要性判斷
import json

def analyze_email_importance(subject, content):
    prompt = f"""
請你扮演一個專門處理個人郵件的 AI 助理。
你任務是判斷「這封郵件是否重要」以及「理由」。

重要性的判斷依據包含但不限於：
- 是否和課程、作業、教務、學校行程有關
- 是否有明確任務、期限、需要行動
- 是否為帳單、金流、報名、活動、提醒
- 是否為個人重要聯絡
- 是否包含緊急或時效性資訊

請用以下 JSON 格式回覆：

{{
  "is_important": true 或 false,
  "reason": "簡述判斷依據"
}}

郵件主旨：
{subject}

郵件內容：
{content}
"""

    response = openai.ChatCompletion.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}]
    )

    result = response.choices[0].message["content"]

    # 安全轉 JSON
    try:
        return json.loads(result)
    except:
        return {"is_important": False, "reason": "AI 回傳格式錯誤"}

產生定期摘要

In [ ]:
#設定時間範圍
from datetime import datetime, timedelta

def get_last_week_range():
    today = datetime.today()
    start = today - timedelta(days=7)
    return start, today

start_date, end_date = get_last_week_range()
start_date, end_date

(datetime.datetime(2026, 2, 4, 9, 26, 34, 664358),
 datetime.datetime(2026, 2, 11, 9, 26, 34, 664358))

In [ ]:
#抓取列表中信箱的7天內郵件
import datetime
import imaplib

# ================= 1. 搜尋函式 (微調版) =================
def fetch_emails_last_week(mail, account_name="未知帳號"):
    """搜尋特定信箱過去 7 天的郵件 ID"""

    # 確保已選擇收件匣
    try:
        status, count = mail.select("inbox")
        if status != 'OK':
            print(f"[{account_name}] 錯誤：無法選擇收件匣")
            return []
    except Exception as e:
        print(f"[{account_name}] 連線異常: {e}")
        return []

    # 設定日期 (過去 7 天)
    start_date = datetime.datetime.now() - datetime.timedelta(days=7)
    date_str = start_date.strftime("%d-%b-%Y") # 格式: 15-Jan-2025

    print(f"[{account_name}] 正在搜尋 {date_str} 之後的郵件...")

    try:
        # 執行搜尋
        type_search, data = mail.search(None, f'(SINCE "{date_str}")')
    except Exception as e:
        print(f"[{account_name}] 搜尋指令失敗: {e}")
        return []

    # 檢查結果
    if type_search != 'OK':
        print(f"[{account_name}] 伺服器拒絕搜尋請求。")
        return []

    if not data or not data[0]:
        print(f"[{account_name}] 結果：過去 7 天無新郵件。")
        return []

    email_ids = data[0].split()
    print(f"[{account_name}] 成功！找到 {len(email_ids)} 封郵件。")

    return email_ids

# ================= 2. 多信箱執行迴圈 =================

# 用來暫存所有信箱的結果，格式: {'Personal': [id1, id2], 'School': [id3]}
all_mailbox_results = {}

# 確保 EMAIL_ACCOUNTS 存在 (來自你之前的設定)
if 'EMAIL_ACCOUNTS' in globals():
    print("啟動多信箱搜尋任務...\n")

    for account in EMAIL_ACCOUNTS:
        name = account['name']
        print(f"--- 處理信箱: {name} ---")

        # 1. 連線 (使用你之前定義的 connect_imap)
        # 如果 connect_imap 沒定義，請把下一行註解掉，先去執行連線函式定義
        if 'connect_imap' in globals():
            mail = connect_imap(account)
        else:
            print("錯誤：找不到 connect_imap 函式，請檢查上一段程式碼。")
            mail = None

        if mail:
            # 2. 呼叫搜尋函式 (傳入 mail 物件 和 帳號名稱)
            ids = fetch_emails_last_week(mail, account_name=name)

            # 3. 儲存結果 (如果有的話)
            if ids:
                all_mailbox_results[name] = {
                    'mail_obj': mail,  # !重要: 暫時保留連線物件以便後續讀取內容
                    'ids': ids
                }
            else:
                # 如果沒信，就直接登出
                try:
                    mail.logout()
                except: pass

        print("\n") # 空一行比較好讀

    print("="*30)
    print(f"所有信箱搜尋完畢。共有 {len(all_mailbox_results)} 個信箱有新郵件。")

else:
    print("錯誤：找不到 EMAIL_ACCOUNTS 設定，請先定義帳號列表。")

啟動多信箱搜尋任務...

--- 處理信箱: Personal (個人) ---
正在連線到 Personal (個人) (dorothea.pudding@gmail.com)...
[Personal (個人)] 正在搜尋 04-Feb-2026 之後的郵件...
[Personal (個人)] 成功！找到 6 封郵件。


--- 處理信箱: Personal1 (個人) ---
正在連線到 Personal1 (個人) (pretty.rubychou@gmail.com)...
[Personal1 (個人)] 正在搜尋 04-Feb-2026 之後的郵件...
[Personal1 (個人)] 成功！找到 6 封郵件。


--- 處理信箱: School (學校) ---
正在連線到 School (學校) (d1134163001@gm.lhu.edu.tw)...
[School (學校)] 正在搜尋 04-Feb-2026 之後的郵件...
[School (學校)] 成功！找到 11 封郵件。


所有信箱搜尋完畢。共有 3 個信箱有新郵件。


In [ ]:
!pip install groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.3/138.3 kB 2.1 MB/s eta 0:00:00


In [ ]:
import os
from groq import Groq

# 設定 API Key
client = Groq(api_key = userdata.get('Groq'))

def analyze_email_importance(subject, content):
    """判斷郵件重要性 (1-5分)"""
    if not content: return "0"

    prompt = f"主旨: {subject}\n內文: {content[:500]}\n請判斷這封信的重要性(1-5分)，只回傳一個數字(例如: 5)。"
    try:
        completion = client.chat.completions.create(
            messages=[{"role": "user", "content": prompt}],
            model="llama-3.3-70b-versatile", # 確保使用最新模型
        )
        return completion.choices[0].message.content.strip()
    except Exception as e:
        print(f"  [AI Error-重要性]: {e}")
        return "0"

def summarize_email(subject, content):
    """產生郵件摘要"""
    if not content: return "無內容可分析"

    prompt = f"主旨: {subject}\n內文: {content[:800]}\n請用繁體中文摘要這封信的重點(50字內)。"
    try:
        completion = client.chat.completions.create(
            messages=[{"role": "user", "content": prompt}],
            model="llama-3.3-70b-versatile", # 確保使用最新模型
        )
        return completion.choices[0].message.content.strip()
    except Exception as e:
        print(f"  [AI Error-摘要]: {e}")
        return "無法產生摘要"

print("AI 分析工具函式已更新。")

AI 分析工具函式已更新。


In [ ]:
#針對每封郵件篩選重要性和產生摘要
import time
import email

def process_emails_in_range(mail, email_ids, account_name="未知帳號"):
    """
    讀取並分析指定 ID 列表的郵件
    參數 account_name:用來標記信件來源 (如 'Personal' 或 'School')
    """
    weekly_data = []

    # 這裡我們不反轉順序，保持傳入時的順序 (通常外部已經反轉過)
    total = len(email_ids)
    print(f"[{account_name}] 準備分析 {total} 封郵件...")

    for index, eid in enumerate(email_ids):
        try:
            # 顯示進度
            print(f"[{account_name}] [{index+1}/{total}] 讀取中...", end="\r")

            # 讀取郵件
            status, msg_data = mail.fetch(eid, "(RFC822)")
            if status != 'OK': continue

            raw_email_bytes = msg_data[0][1]
            msg = email.message_from_bytes(raw_email_bytes)

            # 解析標題與內文 (呼叫之前定義好的工具)
            subject, text_content = extract_subject_and_text(msg)

            # 若內文為空則跳過
            if not text_content or len(text_content.strip()) == 0:
                # print(f"[{account_name}] 跳過 (無內容): {subject[:10]}...")
                continue

            # AI 分析
            importance = analyze_email_importance(subject, text_content)
            summary = summarize_email(subject, text_content)

            # 印出簡單進度 (可選)
            # print(f"[{account_name}] 完成: {subject[:15]}... (分數: {importance})")

            # 存入資料 (新增 'source' 欄位)
            weekly_data.append({
                "source": account_name, # <--- 關鍵修正：標記來源
                "id": eid.decode(),
                "subject": subject,
                "content": text_content,
                "importance": importance,
                "summary": summary
            })

            # 避免 API 速率限制
            time.sleep(0.5)

        except Exception as e:
            print(f"\n[{account_name}] 第 {index+1} 封處理錯誤: {e}")
            continue

    print(f"\n[{account_name}] 分析完成，共產出 {len(weekly_data)} 筆資料。")
    return weekly_data

print("郵件處理函式 process_emails_in_range 已更新 (支援來源標記)。")

郵件處理函式 process_emails_in_range 已更新 (支援來源標記)。


In [ ]:
def generate_weekly_report(weekly_data):
    """
    根據所有信箱的資料生成一份綜合週報
    """
    important_items = []
    normal_items = []

    for item in weekly_data:
        # 1. 取得分數
        try:
            score = int(item["importance"])
        except:
            score = 0

        # 2. 關鍵修正：加入 [來源] 標籤 (例如 [School] 或 [Personal])
        source_tag = f"[{item.get('source', '未知')}]"
        entry_text = f"{source_tag} 標題：{item['subject']} | 摘要：{item['summary']}"

        # 3. 分類
        if score >= 4:
            important_items.append(entry_text)
        else:
            normal_items.append(entry_text)

    # 4. 構建 Prompt
    prompt = f"""
    請你根據以下多個信箱的郵件摘要，製作一份「本週綜合郵件週報」。

    [重要郵件 (High Priority)]
    {"\n".join(important_items) if important_items else "無"}

    [一般郵件 (Normal)]
    {"\n".join(normal_items) if normal_items else "無"}

    週報格式要求：
    1. 【本週重要事件總覽】：條列出最重要的 3-5 件事，並標註來源。
    2. 【需採取行動事項】：條列出需要回覆或處理的事項 (有期限的放前面)。
    3. 【各信箱重點摘要】：
       - 請依據 [Personal] 和 [School] 分類呈現。

    請產生繁體中文的最終週報文本。
    """

    # 5. 使用 Groq 呼叫 AI
    try:
        completion = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[
                {"role": "system", "content": "你是一個專業的行政助手，擅長整理多來源的資訊。"},
                {"role": "user", "content": prompt}
            ]
        )
        return completion.choices[0].message.content
    except Exception as e:
        return f"生成週報時發生錯誤: {e}"

print("週報生成函式 generate_weekly_report 已更新 (支援多來源標示)。")

週報生成函式 generate_weekly_report 已更新 (支援多來源標示)。


In [ ]:
# ==========================================
# 第三部分：執行多信箱自動化流程 (修正 NameError)
# ==========================================

# 1. 準備一個空列表來裝所有信箱的資料
all_weekly_data = []

# 檢查帳號設定是否存在
if 'EMAIL_ACCOUNTS' in globals():
    print("🚀 啟動多信箱週報系統...\n")

    # --- 階段 1: 遍歷所有帳號收集資料 ---
    for account in EMAIL_ACCOUNTS:
        name = account['name']
        print(f"--- 正在檢查: {name} ---")

        # 連線
        mail = connect_imap(account)

        if mail:
            # 搜尋過去 7 天郵件
            email_ids = fetch_emails_last_week(mail, account_name=name)

            if email_ids:
                print(f"[{name}] 發現 {len(email_ids)} 封郵件，開始分析...")

                # 分析郵件 (這會回傳一個列表)
                # 注意：這裡呼叫的是上一段定義好的 process_emails_in_range
                account_data = process_emails_in_range(mail, email_ids, account_name=name)

                # 將結果合併到總表
                all_weekly_data.extend(account_data)

            # 登出該帳號
            try:
                mail.close()
                mail.logout()
            except: pass

        print("\n") # 空一行區隔

    # --- 階段 2: 資料收集完畢，生成報告 ---
    print("="*30)
    print(f"所有信箱檢查完畢。共收集到 {len(all_weekly_data)} 筆有效資料。")

    # 只有當有資料時，才執行生成週報
    if all_weekly_data:
        print("正在生成最終綜合週報 (請稍候)...")

        # 這行就是解決你報錯的關鍵：我們把收集到的 all_weekly_data 傳進去
        final_report = generate_weekly_report(all_weekly_data)

        print("\n" + "="*20 + " 最終週報 " + "="*20 + "\n")
        print(final_report)

        # --- 階段 3: LINE 推播 (如果有設定) ---
        if 'send_line_notify' in globals():
            print("\n正在發送 LINE 通知...")
            send_line_notify("\n" + final_report)
    else:
        print("本週沒有任何需要處理的郵件，因此不生成週報。")

else:
    print("錯誤：找不到 EMAIL_ACCOUNTS 設定，請先執行定義帳號列表的儲存格。")

🚀 啟動多信箱週報系統...

--- 正在檢查: Personal (個人) ---
正在連線到 Personal (個人) (dorothea.pudding@gmail.com)...
[Personal (個人)] 正在搜尋 04-Feb-2026 之後的郵件...
[Personal (個人)] 成功！找到 6 封郵件。
[Personal (個人)] 發現 6 封郵件，開始分析...
[Personal (個人)] 準備分析 6 封郵件...

[Personal (個人)] 分析完成，共產出 6 筆資料。


--- 正在檢查: Personal1 (個人) ---
正在連線到 Personal1 (個人) (pretty.rubychou@gmail.com)...
[Personal1 (個人)] 正在搜尋 04-Feb-2026 之後的郵件...
[Personal1 (個人)] 成功！找到 6 封郵件。
[Personal1 (個人)] 發現 6 封郵件，開始分析...
[Personal1 (個人)] 準備分析 6 封郵件...

[Personal1 (個人)] 分析完成，共產出 6 筆資料。


--- 正在檢查: School (學校) ---
正在連線到 School (學校) (d1134163001@gm.lhu.edu.tw)...
[School (學校)] 正在搜尋 04-Feb-2026 之後的郵件...
[School (學校)] 成功！找到 11 封郵件。
[School (學校)] 發現 11 封郵件，開始分析...
[School (學校)] 準備分析 11 封郵件...

[School (學校)] 分析完成，共產出 11 筆資料。


所有信箱檢查完畢。共收集到 23 筆有效資料。
正在生成最終綜合週報 (請稍候)...

==================== 最終週報 ====================

【本週綜合郵件週報】

### 【本週重要事件總覽】
1. **LINE官方帳號的一對一聊天功能**：可以用來鎖定高價值顧客（來源：Personal）。
2. **PVQC PELC行業專業英文詞彙與聽寫能力國際證書**：您已通過申請，將收到證書（來源：Personal）。
3. **「人工智慧在

line推播通知

In [ ]:
import requests

In [ ]:
from google.colab import userdata
LINE_CHANNEL_ACCESS_TOKEN = userdata.get('LINE_channel')
LINE_USER_ID = userdata.get('LINE_userID')

In [ ]:
def push_line_message(user_id, message):
    url = "https://api.line.me/v2/bot/message/push"

    headers = {
        "Authorization": f"Bearer {LINE_CHANNEL_ACCESS_TOKEN}",
        "Content-Type": "application/json"
    }

    payload = {
        "to": user_id,
        "messages": [
            {
                "type": "text",
                "text": message
            }
        ]
    }

    response = requests.post(url, headers=headers, json=payload)
    return response.status_code, response.text


In [ ]:
message = "📊 本週 AI 郵件週報\n\n" + final_report
status, result = push_line_message(
    LINE_USER_ID,
    message
)

status

200

流程自動化

In [ ]:
#取得今天星期幾
from datetime import datetime

today = datetime.today()
today.weekday()

2

In [ ]:
#排程判斷
def should_generate_weekly_report(target_weekday=6):
    """
    target_weekday:
    0 = 週一, ..., 6 = 週日
    """
    today = datetime.today()
    return today.weekday() == target_weekday

In [ ]:
#排程邏輯測試
if should_generate_weekly_report(6):
    print("今天是週報日，可以產生週報")
else:
    print("今天不是週報日，不執行")

今天不是週報日，不執行


In [ ]:
#主要控制流程
def weekly_job():
    if not should_generate_weekly_report(6):
        print("⏭️ 今天不是週報日，略過")
        return

    print("📊 開始產生 AI 郵件週報...")

    # 你前面已完成的流程
    weekly_data = process_weekly_emails(mail, email_ids)
    weekly_report = generate_weekly_report(weekly_data)

    push_line_message(
        LINE_USER_ID,
        "📊 本週 AI 郵件週報\n\n" + weekly_report
    )

    print("✅ 週報已推播")


In [ ]:
#人工整理需求判斷
def need_manual_review(weekly_data, threshold=5):
    important_count = sum(
        1 for item in weekly_data
        if item["importance"]["is_important"]
    )
    return important_count >= threshold

In [ ]:
#人工整理提醒推播
def send_manual_review_reminder(weekly_data):
    if need_manual_review(weekly_data):
        push_line_message(
            LINE_USER_ID,
            "⚠️ 本週重要郵件較多，建議你手動整理與確認。"
        )

In [ ]:
import datetime

# ================= 1. 輔助函式定義 =================

def should_generate_weekly_report(target_weekday=6):
    """
    檢查今天是否為指定星期 (0=週一, 6=週日)
    預設為 6 (週日)
    """
    today = datetime.datetime.now().weekday()
    return today == target_weekday

def send_manual_review_reminder(data):
    """
    發送人工審核提醒 (列出標題清單)
    讓你知道系統抓到了哪些信，作為生成前的確認
    """
    if not data: return

    titles = [f"- [{item['source']}] {item['subject']}" for item in data]
    msg = "\n🔍 [系統通知] 資料收集完成，準備生成週報。\n本週抓取到的郵件清單：\n" + "\n".join(titles)

    # 傳送到 LINE
    if 'send_line_notify' in globals():
        send_line_notify(msg)
    print("已發送資料清單確認通知。")

# ================= 2. 主任務流程 =================

def weekly_job():
    print("⏰ 開始執行週報排程檢查...")

    # 1. 檢查日期 (這裡設 6 代表週日，測試時可暫時註解掉這兩行)
    if not should_generate_weekly_report(6):
         print("⏭️ 今天不是週報日，略過任務。")
         return

    # 2. 收集資料 (整合之前的多信箱迴圈邏輯)
    all_weekly_data = []

    if 'EMAIL_ACCOUNTS' in globals():
        for account in EMAIL_ACCOUNTS:
            name = account['name']
            print(f"--- 正在檢查: {name} ---")

            mail = connect_imap(account)
            if mail:
                email_ids = fetch_emails_last_week(mail, account_name=name)
                if email_ids:
                    # 使用前面定義好的 process_emails_in_range
                    acc_data = process_emails_in_range(mail, email_ids, account_name=name)
                    all_weekly_data.extend(acc_data)

                try:
                    mail.close()
                    mail.logout()
                except: pass
            print("\n")
    else:
        print("❌ 錯誤：找不到帳號設定。")
        return

    # 3. 處理資料與發送
    if all_weekly_data:
        print(f"✅ 資料收集完畢，共 {len(all_weekly_data)} 筆。")

        # A. 發送人工整理提醒 (通知你已抓到資料)
        send_manual_review_reminder(all_weekly_data)

        # B. 生成 AI 週報
        print("📊 正在生成 AI 週報...")
        weekly_report = generate_weekly_report(all_weekly_data)

        # C. 發送最終週報
        print("🚀 發送 LINE 通知...")
        if 'send_line_notify' in globals():
            send_line_notify("\n📊 本週 AI 郵件週報\n\n" + weekly_report)

        print("✅ 週報任務全數完成！")

    else:
        print("📭 本週無新郵件資料，不需生成週報。")
        if 'send_line_notify' in globals():
            send_line_notify("📭 本週無重要郵件，無需生成週報。")

# ================= 3. 立即測試 =================
# 執行函式來測試流程
weekly_job()

⏰ 開始執行週報排程檢查...
⏭️ 今天不是週報日，略過任務。


我的寒假延伸學習（暫定）：
* 製作使用說明
* 流程完全自動化
* LINE個人通知／簡易軟體（擇一，暫定用LINE）
